# Chapter 3 Practical 06: Cold Start, Sparsity, Clustering, and Temporal Dynamics

Learning objectives:
- Detect cold-start users and items.
- Apply minimum-overlap and shrinkage.
- Cluster users to reduce neighbor search.
- Add time-decay weights to recent interactions.

Slide connection: limits of memory-based CF, practical mitigations, clustering for scalability, and temporal dynamics.


In [1]:
# Teaching note: Load ratings and movies with a Colab-safe fallback. Local files are used first; GitHub raw CSVs are used when opened directly from GitHub.
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
# Pivot interactions into rows = users and columns = items.
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Loaded ratings_chapter3.csv from data/ratings_chapter3.csv
Loaded movies_chapter3.csv from data/movies_chapter3.csv


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
user_id,,,,,,,,,,
Alice,5.0,NaN,4.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN
Bob,NaN,NaN,6.0,4.0,7.0,4.0,7.0,NaN,NaN,NaN
Chris,NaN,NaN,2.0,7.0,3.0,7.0,NaN,NaN,NaN,5.0
Karen,NaN,NaN,NaN,4.0,7.0,3.0,6.0,NaN,NaN,NaN
Lynn,NaN,NaN,2.0,4.0,4.0,6.0,NaN,NaN,NaN,6.0
Nina,NaN,5.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,5.0
Omar,NaN,2.0,NaN,3.0,NaN,NaN,NaN,5.0,5.0,NaN
Sally,NaN,NaN,7.0,6.0,7.0,3.0,6.0,NaN,NaN,NaN


In [2]:
# Teaching note: Identify users or items with too few interactions for reliable CF.
# Groupby aggregates ratings by user or item for summary statistics.
user_counts = ratings_named.groupby("user_id").size().rename("ratings_count")
# Groupby aggregates ratings by user or item for summary statistics.
item_counts = ratings_named.groupby("title").size().rename("ratings_count")

print("Cold-start-like users:")
display(user_counts[user_counts <= 2])
print("Cold-start-like items:")
display(item_counts[item_counts <= 2])


Cold-start-like users:


Series([], Name: ratings_count, dtype: int64)

Cold-start-like items:


title
Blade Runner    1
Finding Nemo    2
The Notebook    1
Titanic         2
Name: ratings_count, dtype: int64

In [3]:
# Teaching note: Use shrinkage and minimum overlap to reduce noisy similarities.
# Minimum overlap and shrinkage make sparse similarities more stable.
def pearson_with_shrinkage(matrix, user_a, user_b, min_overlap=2, alpha=3):
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    overlap = pair.shape[1]
    if overlap < min_overlap:
        return np.nan
    if pair.loc[user_a].std() == 0 or pair.loc[user_b].std() == 0:
        return np.nan
    raw = np.corrcoef(pair.loc[user_a], pair.loc[user_b])[0, 1]
    return raw * overlap / (overlap + alpha)

rows = []
for other in rating_matrix.index.drop("Karen"):
    rows.append({
        "neighbor": other,
        "shrunk_similarity": pearson_with_shrinkage(rating_matrix, "Karen", other),
    })
pd.DataFrame(rows).sort_values("shrunk_similarity", ascending=False).round(3)


,neighbor,shrunk_similarity
1,Bob,0.542
6,Sally,0.482
3,Lynn,-0.347
0,Alice,-0.400
2,Chris,-0.485
4,Nina,NaN
5,Omar,NaN


In [4]:
# Teaching note: Cluster users after filling missing values to reduce neighbor search space.
# K-Means groups similar users so we can search a smaller neighborhood.
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

filled = rating_matrix.apply(lambda col: col.fillna(col.mean()), axis=0)
scaled = StandardScaler().fit_transform(filled)
# K-Means groups similar users so we can search a smaller neighborhood.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = pd.Series(kmeans.fit_predict(scaled), index=rating_matrix.index, name="cluster")
clusters.to_frame().sort_values("cluster")


,cluster
user_id,
Omar,0
Alice,1
Chris,1
Lynn,1
Bob,2
Karen,2
Nina,2
Sally,2


In [5]:
# Teaching note: Select only neighbors from the target user cluster for faster recommendation.
target_user = "Karen"
target_cluster = clusters.loc[target_user]
candidate_neighbors = clusters[clusters.eq(target_cluster)].index.drop(target_user)
print(f"Compare Karen only with users in cluster {target_cluster}: {candidate_neighbors.tolist()}")


Compare Karen only with users in cluster 2: ['Bob', 'Nina', 'Sally']


In [6]:
# Teaching note: Compute exponential time-decay weights so recent ratings matter more.
decay_lambda = 0.01
# Exponential decay gives recent interactions larger weights than old ones.
ratings_named["time_weight"] = np.exp(-decay_lambda * ratings_named["days_ago"])
ratings_named["weighted_rating"] = ratings_named["rating"] * ratings_named["time_weight"]

ratings_named[["user_id", "title", "rating", "days_ago", "time_weight", "weighted_rating"]].sort_values("days_ago").head(10).round(3)


,user_id,title,rating,days_ago,time_weight,weighted_rating
33,Omar,The Notebook,5,6,0.942,4.709
9,Bob,The Matrix,7,6,0.942,6.592
29,Nina,Finding Nemo,5,7,0.932,4.662
23,Karen,The Matrix,6,8,0.923,5.539
19,Lynn,Toy Story,6,10,0.905,5.429
18,Lynn,Independence Day,2,12,0.887,1.774
25,Alice,Blade Runner,5,12,0.887,4.435
4,Sally,The Matrix,6,14,0.869,5.216
32,Omar,Titanic,5,15,0.861,4.304
3,Sally,Independence Day,7,20,0.819,5.731


In [7]:
# Teaching note: Create a time-weighted item popularity signal.
recent_profile = (
    # Groupby aggregates ratings by user or item for summary statistics.
    ratings_named.groupby("title")
    .apply(lambda g: np.average(g["rating"], weights=g["time_weight"]))
    .rename("time_weighted_mean")
    .sort_values(ascending=False)
)
recent_profile.head(5).round(2)


/var/folders/w0/2jgnn0bx27b_jyy4mrm45cxm0000gn/T/ipykernel_2602/2606139541.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ratings_named.groupby("title")


title
The Matrix      6.08
Toy Story       5.37
Blade Runner    5.00
The Notebook    5.00
Star Wars       4.77
Name: time_weighted_mean, dtype: float64

## Challenge Lab

1. Increase `decay_lambda` and recompute the time-weighted item means. Which movies become more important because of recent ratings?
2. Change the number of K-Means clusters and inspect `candidate_neighbors` for `Karen`. Explain how clustering changes the search space for memory-based CF.
